<a href="https://colab.research.google.com/github/ekonishi8524/my-colab-notebooks/blob/main/cifar10_autoencoder_anomaly_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

In [ ]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_indices = [i for i, (_, label) in enumerate(train_set) if label == 0]
train_subset = torch.utils.data.Subset(train_set, train_indices)
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)

In [ ]:
class StrippedAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_encode = nn.Linear(64 * 8 * 8, 16)
        self.fc_decode = nn.Linear(16, 64 * 8 * 8)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 2, stride=2), nn.Tanh()
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.view(x.size(0), -1)
        latent = torch.relu(self.fc_encode(x))
        x = torch.relu(self.fc_decode(latent))
        x = x.view(x.size(0), 64, 8, 8)
        return self.decoder(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StrippedAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
from tqdm import tqdm

print("Training on Airplanes...")
NUM_EPOCHS = 15
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for data, _ in tqdm(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, data)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Avg Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
def get_anomaly_score(img_tensor):
    model.eval()
    with torch.no_grad():
        img_tensor = img_tensor.to(device).unsqueeze(0)
        recon = model(img_tensor)
        return criterion(recon, img_tensor).item()

test_normal_img, _ = test_set[[i for i, (_, l) in enumerate(test_set) if l == 0][0]]
test_anomaly_img, _ = test_set[[i for i, (_, l) in enumerate(test_set) if l == 5][0]]

print(f"Normal (Airplane) Score: {get_anomaly_score(test_normal_img):.4f}")
print(f"Anomaly (Dog) Score: {get_anomaly_score(test_anomaly_img):.4f}")

In [ ]:
def visualize_top_anomalies(model, dataset, device, num_images=4):
    model.eval()
    scores = []

    print("Calculating Scores...")
    with torch.no_grad():
        for i in range(len(dataset)):
            img, _ = dataset[i]
            input_tensor = img.to(device).unsqueeze(0)
            reconstructed = model(input_tensor)

            loss = torch.nn.functional.mse_loss(input_tensor, reconstructed)
            scores.append((i, loss.item()))

    scores.sort(key=lambda x: x[1], reverse=True)

    top_indices = [x[0] for x in scores[:num_images]]
    top_scores = [x[1] for x in scores[:num_images]]

    print(f"Worst Scores: {top_indices}")

    fig, axes = plt.subplots(num_images, 3, figsize=(10, 3 * num_images))

    for i, idx in enumerate(top_indices):
        img, label = dataset[idx]
        input_tensor = img.to(device).unsqueeze(0)
        reconstructed = model(input_tensor)

        img_in = (img.permute(1, 2, 0).cpu().detach().numpy() + 1) / 2
        img_out = (reconstructed.squeeze().permute(1, 2, 0).cpu().detach().numpy() + 1) / 2
        diff = torch.abs(input_tensor - reconstructed)
        anomaly_map = torch.mean(diff, dim=1).squeeze().cpu().detach().numpy()

        axes[i, 0].imshow(img_in.clip(0, 1))
        axes[i, 1].imshow(img_out.clip(0, 1))
        axes[i, 2].imshow(anomaly_map, cmap='jet')

        axes[i, 0].set_ylabel(f"Idx: {idx}\nScore: {top_scores[i]:.4f}", fontsize=10)
        for j in range(3):
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])

    plt.tight_layout()
    plt.show()

visualize_top_anomalies(model, test_set, device, num_images=4)

In [ ]:
import numpy as np

def calculate_threshold(model, normal_dataloader, device):
    model.eval()
    normal_losses = []

    with torch.no_grad():
        for imgs, _ in normal_dataloader:
            imgs = imgs.to(device)
            reconstructed = model(imgs)
            loss = torch.nn.functional.mse_loss(
                reconstructed, imgs, reduction='none'
            ).mean([1, 2, 3])
            normal_losses.extend(loss.cpu().numpy())

    mu = np.mean(normal_losses)
    std = np.std(normal_losses)
    threshold = mu + 3 * std

    print(f"μ: {mu:.4f}")
    print(f"μ+3σ: {threshold:.4f}")
    return threshold

def predict_anomaly(model, img, threshold, device):

    model.eval()
    with torch.no_grad():
        input_tensor = img.to(device).unsqueeze(0)
        reconstructed = model(input_tensor)
        loss = torch.nn.functional.mse_loss(input_tensor, reconstructed).item()

        is_anomaly = loss > threshold
        status = "ANOMALY" if is_anomaly else "NORMAL"
        print(f"Score: {loss:.4f} | Result: {status}")
        return is_anomaly

threshold = calculate_threshold(model, train_loader, device)
test_image, _ = test_set[0]
predict_anomaly(model, test_image, threshold, device)

In [ ]:
from sklearn.metrics import roc_curve, auc

def plot_roc_curve(model, test_loader, device):
    model.eval()
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in test_loader:
            true_binary_labels = (labels != 0).int()

            imgs = imgs.to(device)
            reconstructed = model(imgs)

            loss = torch.nn.functional.mse_loss(
                reconstructed, imgs, reduction='none'
            ).view(imgs.size(0), -1).mean(dim=1)

            all_scores.extend(loss.cpu().numpy())
            all_labels.extend(true_binary_labels.numpy())

    fpr, tpr, thresholds = roc_curve(all_labels, all_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

test_loader = DataLoader(test_set, batch_size=64, shuffle=False)
plot_roc_curve(model, test_loader, device)